In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# 1. Load your dataset
df = pd.read_csv("C:/Users/woons/Documents/Python Projects/capstone_part2/part3_machine learning/data/features_traffic_proxy.csv")

print("==========================================")
print("1. K-MEANS CLUSTERING ON TRAFFIC CONDITIONS")
print("==========================================")

# Create a meaningful ordinal weather severity score based on weather_main
# Adjust the mapping dictionary to match your dataset's specific weather categories if needed
severity_mapping = {
    'Clear': 0,
    'Clouds': 1,
    'Mist': 1,
    'Fog': 2,
    'Haze': 2,
    'Drizzle': 2,
    'Rain': 3,
    'Snow': 3,
    'Thunderstorm': 4,
    'Squall': 4
}
# Fallback to 1 if a category isn't in the mapping
df['weather_severity'] = df['weather_main'].map(severity_mapping).fillna(1)

# Use 'weather_severity' instead of raw 'weather_main_code'
cluster_features = ['traffic_volume', 'temp_scaled', 'clouds_all_scaled', 'hour', 'weather_severity']
X_cluster = df[cluster_features].dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
X_cluster['cluster'] = kmeans.fit_predict(X_scaled)

cluster_summary = X_cluster.groupby('cluster').mean()
print("\nCluster Profiles (Mean Values):")
print(cluster_summary.to_string())

print("\n==========================================")
print("2. ASSOCIATION RULE MINING (Apriori)")
print("==========================================")

df_rules = df.copy()

def bin_hour(h):
    if 6 <= h < 10:
        return "Morning_Rush"
    elif 10 <= h < 16:
        return "Midday"
    elif 16 <= h < 20:
        return "Evening_Rush"
    else:
        return "Night"

df_rules['time_period'] = df_rules['hour'].apply(bin_hour)
df_rules['day_type'] = df_rules['is_weekend'].apply(lambda x: "Weekend" if x == 1 else "Weekday")
df_rules['weather_cond'] = df_rules['weather_main']
df_rules['congestion'] = df_rules['congestion_category']

transaction_cols = ['time_period', 'day_type', 'weather_cond', 'congestion']
transactions = df_rules[transaction_cols].astype(str).values.tolist()

te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

frequent_itemsets = apriori(df_encoded, min_support=0.03, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

congestion_labels = ['Low', 'Moderate', 'High', 'Severe']
congestion_rules = rules[rules['consequents'].apply(lambda x: any(label in list(x) for label in congestion_labels))].copy()

top_rules = congestion_rules.sort_values(by='lift', ascending=False).head(5)

print("\nTop Association Rules Predicting Congestion (by highest lift):")
for idx, row in top_rules.reset_index().iterrows():
    print(f"Rule {idx + 1}: {list(row['antecedents'])} ==> {list(row['consequents'])}")
    print(f"   Support: {row['support']:.4f} | Confidence: {row['confidence']:.4f} | Lift: {row['lift']:.4f}\n")

1. K-MEANS CLUSTERING ON TRAFFIC CONDITIONS

Cluster Profiles (Mean Values):
         traffic_volume  temp_scaled  clouds_all_scaled       hour  weather_severity
cluster                                                                             
0           4240.570712     0.549199           0.812825  14.715564          1.812337
1           4255.850984     0.632481           0.108157  15.146476          0.450434
2            988.106988     0.532766           0.061324   3.196217          0.399076
3            929.018973     0.527520           0.787918   3.278625          1.926363

2. ASSOCIATION RULE MINING (Apriori)

Top Association Rules Predicting Congestion (by highest lift):
Rule 1: ['Morning_Rush', 'Weekday'] ==> ['Severe']
   Support: 0.1003 | Confidence: 0.8217 | Lift: 3.2894

Rule 2: ['Weekend', 'Midday'] ==> ['High']
   Support: 0.0557 | Confidence: 0.8044 | Lift: 3.2156

Rule 3: ['Evening_Rush', 'Weekend'] ==> ['High']
   Support: 0.0338 | Confidence: 0.7330 | Lift: 2.9302

